## Importing Libraries

In [53]:
import pandas as pd
import numpy as np
from pathlib import Path
import clickhouse_connect

### Define IMEI Date Filter

In [54]:
IMEI_DATE_FILTER = """
    WHERE imei_first_seen >= '2026-05-01'
      AND imei_first_seen < '2026-06-01'
"""

## Importing KYC data

In [55]:
# 1. Define your base directory using Pathlib (makes it easy to update later)
BASE_DIR = Path("/Volumes/E$/KYC/Merged Clean Dumps/2026")

# 2. Load only two DataFrames to save massive amounts of RAM  ---- Change Dataframe Names ‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️
df = pd.read_parquet(BASE_DIR / "df_04_26.parquet")
df_NID = pd.read_parquet(BASE_DIR / "df_NID_04_26.parquet")

In [56]:
df.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno
0,0700528595,MISAKI,NAMBIGO,NATIONAL_ID,CM950601023PRA,070,AIRTEL
1,0700913847,JOAN,NALUKWAGO,NATIONAL_ID,CF92024105MCZE,070,AIRTEL
2,0750958226,HARRIET,EBOL,NATIONAL_ID,CF90103100DD5D,075,AIRTEL
3,0702338467,BEATRICE,NABWIRE,NATIONAL_ID,CF890671078H3L,070,AIRTEL
4,0752598275,TRACETRACK SYSTEMS LIMITED,UNKNOWN,COMPANY_ID,80020000518498,075,AIRTEL


In [57]:
df_NID.head()

,msisdn,first_name,surname,id_type,id_number,prefix,mno,gender,birth_year,age,district
0,0700528595,MISAKI,NAMBIGO,NATIONAL_ID,CM950601023PRA,070,AIRTEL,Male,1995,31,BUTALEJA
1,0700913847,JOAN,NALUKWAGO,NATIONAL_ID,CF92024105MCZE,070,AIRTEL,Female,1992,34,MASAKA
2,0750958226,HARRIET,EBOL,NATIONAL_ID,CF90103100DD5D,075,AIRTEL,Female,1990,36,KOLE
3,0702338467,BEATRICE,NABWIRE,NATIONAL_ID,CF890671078H3L,070,AIRTEL,Female,1989,37,MANAFWA
4,0756115887,BRIDGET,NANTALE,NATIONAL_ID,CF93036104D3DK,075,AIRTEL,Female,1993,33,RAKAI


In [58]:
# Keep only rows where the MSISDN is exactly 10 characters long
df_NID = df_NID[df_NID['msisdn'].astype(str).str.len() == 10]
df = df[df['msisdn'].astype(str).str.len() == 10]

# Ensure the column is treated as text, then replace the leading '0' with '256'
df_NID['msisdn'] = df_NID['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)
df['msisdn'] = df['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)


# Drop unecessary columns to save memory
df = df.drop(columns=['surname', 'first_name'])
df_NID = df_NID.drop(columns=['surname', 'first_name'])

## Importing GSMA data

In [59]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the gsma_devices table
gsma_query = """
SELECT
    tac,
    oem,
    brand,
    model,
    marketing_name,
    device_type,
    os_family,
    os_version,
    sim_slots,
    has_2g,
    has_3g,
    has_4g,
    has_5g,
    year_released
FROM gsma_devices
"""
gsma_df = client.query_df(gsma_query)

In [60]:
len(gsma_df)

290402

In [61]:
gsma_df.head()

,tac,oem,brand,model,marketing_name,device_type,os_family,os_version,sim_slots,has_2g,has_3g,has_4g,has_5g,year_released
0,35697403,Not Known,Not Known,ROWEL K658,,Handheld,,,0,0,0,0,0,0
1,35697404,Not Known,G crown,"G265, G765, G865, G965",,Handheld,,,0,0,0,0,0,0
2,35697405,Not Known,QMobile,Q4,Q4 TV,Handheld,Other,,0,1,0,0,0,2013
3,35697406,Not Known,Apple,iPad mini (A1600),iPad mini 3,Tablet,iOS,8_1,1,1,1,1,0,2014
4,35697407,Not Known,TC,TC F6,,Mobile Phone/Feature phone,,,0,0,0,0,0,0


## Importing Fake IMEIs Table

In [62]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_memory_usage': 2000000000,  # 2GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the imeis_fake table
fake_query = f"SELECT * FROM imeis_fake_v {IMEI_DATE_FILTER}"

fake_df = client.query_df(fake_query)

In [63]:
len(fake_df)

250559

In [64]:
fake_df.head(100)

,imei,last_seen,imei_status,imsi,msisdn,device_type,imei_first_seen,cgi,rat,core_type
0,00000000166285,2026-05-22 14:45:45,W,641101970878456,,<NA>,2026-05-25 18:32:33,,,
1,00000000432125,2026-05-22 17:03:57,W,641101974606362,256779080989,<NA>,2026-05-22 08:59:56,,,
2,00000000528934,2026-05-23 13:38:49,W,641010409065933,256703588730,<NA>,2026-05-29 05:51:52,641-01-7064-54562,6,
3,00000011111111,2026-05-22 14:15:58,W,641101945475179,256783647367,<NA>,2026-05-22 11:21:03,,,
4,00000355474567,2026-05-22 16:33:21,W,641101974717663,,<NA>,2026-05-23 19:07:33,,,
...,...,...,...,...,...,...,...,...,...,...
95,05845321435522,2026-05-22 17:25:03,W,641101926540577,256788909062,<NA>,2026-05-22 07:56:07,,,
96,05875424325368,2026-05-22 17:23:39,W,641010231816770,256705836141,<NA>,2026-05-27 19:55:11,,,
97,05876432457680,2026-05-22 18:12:25,W,641101913104253,256784794859,<NA>,2026-05-22 11:01:11,,,
98,05876867564221,2026-05-22 14:09:56,W,641101972671138,256764075616,<NA>,2026-05-22 09:50:28,,,


In [65]:
# Drop Empty Device Type Column
fake_df = fake_df.drop(columns=['device_type'])


# Re Arrange Columns
fake_df=fake_df[['imei_first_seen', 'last_seen', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

In [66]:
# Step 1: attach the rich NID KYC (gender, age, district) where msisdn is NID-registered
fake_df = fake_df.merge(df_NID, on='msisdn', how='left')

# Step 2: bring in df for the fallback, using suffixes to keep the collisions separate
fake_df = fake_df.merge(df, on='msisdn', how='left', suffixes=('', '_df'))

# Step 3: coalesce — df_NID value wins, df fills only where df_NID was NaN
overlap = ['id_type', 'id_number', 'prefix', 'mno']  # columns that exist in both df_NID and df
for col in overlap:
    target = fake_df[col].astype('object')        # drop categorical restriction
    source = fake_df[f'{col}_df'].astype('object')
    fake_df[col] = target.fillna(source)

# Step 4: drop the now-redundant _df columns
fake_df = fake_df.drop(columns=[f'{col}_df' for col in overlap])

# Step 5: convert the relevant columns back to categorical for memory efficiency
for col in ['id_type', 'prefix', 'mno']:
    fake_df[col] = fake_df[col].astype('category')

In [67]:
# generate mno from imsi where missing, using the standard prefix mapping
# ======================================================================
# Only touch rows where mno is currently missing
mask = fake_df['mno'].isna()

# Make sure imsi is string so .str works reliably
imsi = fake_df['imsi'].astype('string')

# Derive operator from the IMSI prefix
mtn    = mask & imsi.str.startswith('64110')
hamilton    = mask & imsi.str.startswith('64120')
talkio = mask & imsi.str.startswith('64108')
airtel = mask & (imsi.str.startswith('64101') | imsi.str.startswith('64122'))

# If mno is categorical, add the new categories before assigning (same trap as before)
if isinstance(fake_df['mno'].dtype, pd.CategoricalDtype):
    fake_df['mno'] = fake_df['mno'].cat.add_categories(
        [c for c in ['MTN', 'AIRTEL', 'HAMILTON', 'TALKIO'] if c not in fake_df['mno'].cat.categories]
    )

fake_df.loc[mtn, 'mno']    = 'MTN'
fake_df.loc[hamilton, 'mno'] = 'HAMILTON'
fake_df.loc[talkio, 'mno'] = 'TALKIO'
fake_df.loc[airtel, 'mno'] = 'AIRTEL'

In [68]:
# Enrich Dataset with MCC - to Have Country Information for each IMSI

# 1) Load MCC lookup table
mcc_lu = pd.read_csv("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/MCC_Each_country.csv", dtype=str)
mcc_lu.head()


# 2) Normalize lookup columns
mcc_lu = mcc_lu.rename(columns={"MCC": "mcc", "Country": "country"})
mcc_lu["mcc"] = mcc_lu["mcc"].astype("string").str.strip()
mcc_lu["country"] = mcc_lu["country"].astype("string").str.strip()
mcc_lu = mcc_lu.drop_duplicates(subset=["mcc"])


# 3) Extract MCC from IMSI (first 3 digits)
fake_df["imsi"] = fake_df["imsi"].astype("string")

fake_df["mcc"] = (
    fake_df["imsi"]
    .str.replace(r"\D+", "", regex=True)  # keep digits only
    .str.slice(0, 3)
)


# 4) Merge Country Code Data Frame with the Roaming Data sets
fake_df = fake_df.merge(
    mcc_lu[["mcc", "country"]],
    how="left",
    on="mcc"
)


# 5) Optional: fill unknowns
fake_df["country"] = fake_df["country"].fillna("UNKNOWN")


In [69]:
fake_df.head()

,imei_first_seen,last_seen,imei,imei_status,imsi,msisdn,rat,cgi,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2026-05-25 18:32:33,2026-05-22 14:45:45,00000000166285,W,641101970878456,,,,NaN,<NA>,NaN,MTN,NaN,<NA>,<NA>,NaN,641,Uganda
1,2026-05-22 08:59:56,2026-05-22 17:03:57,00000000432125,W,641101974606362,256779080989,,,NATIONAL_ID,CM02003105VMDL,077,MTN,Male,2002,24,BUNDIBUGYO,641,Uganda
2,2026-05-29 05:51:52,2026-05-23 13:38:49,00000000528934,W,641010409065933,256703588730,6,641-01-7064-54562,NATIONAL_ID,CF68049101JA6J,070,AIRTEL,Female,1968,58,MAYUGE,641,Uganda
3,2026-05-22 11:21:03,2026-05-22 14:15:58,00000011111111,W,641101945475179,256783647367,,,NATIONAL_ID,CF6200910496ZC,078,MTN,Female,1962,64,KABALE,641,Uganda
4,2026-05-23 19:07:33,2026-05-22 16:33:21,00000355474567,W,641101974717663,,,,NaN,<NA>,NaN,MTN,NaN,<NA>,<NA>,NaN,641,Uganda


## Importing Genuine IMEIs Table

### Change to Eventually Pick only one month !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [70]:
# Connect to ClickHouse with resource limits
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    database='ceir_gold',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

# Step 1: Fetch data from the imeis_genuine table
genuine_query = f"SELECT * FROM imeis_genuine_v {IMEI_DATE_FILTER}"

genuine_df = client.query_df(genuine_query)

In [71]:
len(genuine_df)

3313834

In [72]:
# Drop Device Type Column - we shall regenerate it from GSMA TAC data
genuine_df = genuine_df.drop(columns=['device_type'])

# Convert IMEI to string, extract the first 8 characters, and create the 'tac' column
genuine_df['tac'] = genuine_df['imei'].astype(str).str[:8]

# Re Arrange Columns
genuine_df=genuine_df[['imei_first_seen','last_seen', 'tac', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

# Merge with GSMA Data to get device details for fake IMEIs
genuine_df = genuine_df.merge(gsma_df, on='tac', how='left')

In [73]:
# Step 1: attach the rich NID KYC (gender, age, district) where msisdn is NID-registered
genuine_df = genuine_df.merge(df_NID, on='msisdn', how='left')

# Step 2: bring in df for the fallback, using suffixes to keep the collisions separate
genuine_df = genuine_df.merge(df, on='msisdn', how='left', suffixes=('', '_df'))

# Step 3: coalesce — df_NID value wins, df fills only where df_NID was NaN
overlap = ['id_type', 'id_number', 'prefix', 'mno']  # columns that exist in both df_NID and df
for col in overlap:
    target = genuine_df[col].astype('object')        # drop categorical restriction
    source = genuine_df[f'{col}_df'].astype('object')
    genuine_df[col] = target.fillna(source)

# Step 4: drop the now-redundant _df columns
genuine_df = genuine_df.drop(columns=[f'{col}_df' for col in overlap])

# Step 5: convert the relevant columns back to categorical for memory efficiency
for col in ['id_type', 'prefix', 'mno']:
    genuine_df[col] = genuine_df[col].astype('category')

In [74]:
# generate mno from imsi where missing, using the standard prefix mapping
# ======================================================================
# Only touch rows where mno is currently missing
mask = genuine_df['mno'].isna()

# Make sure imsi is string so .str works reliably
imsi = genuine_df['imsi'].astype('string')

# Derive operator from the IMSI prefix
mtn    = mask & imsi.str.startswith('64110')
hamilton    = mask & imsi.str.startswith('64120')
talkio = mask & imsi.str.startswith('64108')
airtel = mask & (imsi.str.startswith('64101') | imsi.str.startswith('64122'))

# If mno is categorical, add the new categories before assigning (same trap as before)
if isinstance(genuine_df['mno'].dtype, pd.CategoricalDtype):
    genuine_df['mno'] = genuine_df['mno'].cat.add_categories(
        [c for c in ['MTN', 'AIRTEL', 'HAMILTON', 'TALKIO'] if c not in genuine_df['mno'].cat.categories]
    )

genuine_df.loc[mtn, 'mno']    = 'MTN'
genuine_df.loc[hamilton, 'mno'] = 'HAMILTON'
genuine_df.loc[talkio, 'mno'] = 'TALKIO'
genuine_df.loc[airtel, 'mno'] = 'AIRTEL'

In [75]:
# Enrich Dataset with MCC - to Have Country Information for each IMSI

# 1) Load MCC lookup table
mcc_lu = pd.read_csv("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/MCC_Each_country.csv", dtype=str)
mcc_lu.head()


# 2) Normalize lookup columns
mcc_lu = mcc_lu.rename(columns={"MCC": "mcc", "Country": "country"})
mcc_lu["mcc"] = mcc_lu["mcc"].astype("string").str.strip()
mcc_lu["country"] = mcc_lu["country"].astype("string").str.strip()
mcc_lu = mcc_lu.drop_duplicates(subset=["mcc"])


# 3) Extract MCC from IMSI (first 3 digits)
genuine_df["imsi"] = genuine_df["imsi"].astype("string")

genuine_df["mcc"] = (
    genuine_df["imsi"]
    .str.replace(r"\D+", "", regex=True)  # keep digits only
    .str.slice(0, 3)
)


# 4) Merge Country Code Data Frame with the Roaming Data sets
genuine_df = genuine_df.merge(
    mcc_lu[["mcc", "country"]],
    how="left",
    on="mcc"
)

# 5) Optional: fill unknowns
genuine_df["country"] = genuine_df["country"].fillna("UNKNOWN")


In [76]:
#Change Data type to remove decimal points and convert to integers
genuine_df['sim_slots'] = genuine_df['sim_slots'].astype('Int64')
genuine_df['has_2g'] = genuine_df['has_2g'].astype('Int64')
genuine_df['has_3g'] = genuine_df['has_3g'].astype('Int64')
genuine_df['has_4g'] = genuine_df['has_4g'].astype('Int64')
genuine_df['has_5g'] = genuine_df['has_5g'].astype('Int64')
genuine_df['year_released'] = genuine_df['year_released'].astype('Int64')

In [77]:
genuine_df.head()

,imei_first_seen,last_seen,tac,imei,imei_status,imsi,msisdn,rat,cgi,oem,...,id_type,id_number,prefix,mno,gender,birth_year,age,district,mcc,country
0,2026-05-22 10:22:02,2026-05-22 18:00:41,00440111,00440111639201,W,641101933398605,256762370575,,,Not Known,...,NATIONAL_ID,CM98094101RP9C,076,MTN,Male,1998,28,LUUKA,641,Uganda
1,2026-05-22 07:33:21,2026-05-22 16:43:30,00440111,00440111639340,W,641101924316458,256761438093,,,Not Known,...,NATIONAL_ID,CM0102310K181C,076,MTN,Male,2001,25,LUWEERO,641,Uganda
2,2026-05-23 05:15:02,1970-01-01 00:00:00,00440123,00440123063041,,,,,,Not Known,...,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN,,UNKNOWN
3,2026-05-24 21:22:10,2026-05-22 16:00:40,00440254,00440254260845,W,641101960691280,256763225752,,,Not Known,...,REFUGEE_ID,MMF-00024475,076,MTN,NaN,<NA>,<NA>,NaN,641,Uganda
4,2026-05-22 08:37:41,2026-05-22 16:14:15,00440254,00440254264116,W,641101973686432,,,,Not Known,...,NaN,<NA>,NaN,MTN,NaN,<NA>,<NA>,NaN,641,Uganda


In [78]:
# Export the enriched datasets to Parquet for future use
fake_df.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/fake_2026-05.parquet") # Update File Name as Needed
genuine_df.to_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/genuine_2026-05.parquet") # Update File Name as Needed

## Cloned IMEIs

## PWD Data